# Conceptual questions


Let’s walk through what actually happens—chronologically, and concretely—when we execute the cells below.

I’ll assume the structure:

```text
D:\DigitalTwinsGeneratorGUI\
  generator\
    app.py
    ...
  analyzer\
    app.py
    ...
  config.json   (either existing or created by you)
```

---

### Cell 1 – Path setup

```python
import sys
import os

PROJECT_ROOT = r"D:\DigitalTwinsGeneratorGUI"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)
```

**What happens:**

- Python’s import search path (`sys.path`) is extended with `D:\DigitalTwinsGeneratorGUI`.
- This means `import generator` and `import analyzer` (and their submodules) will work if you ever want to import them directly in the notebook.
- No files are created or modified here—this is purely in‑memory configuration for the current Jupyter kernel.

---

### Cell 2 – Dependency installation

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

**What happens:**

- `pip` installs (or confirms) the required packages into the environment where the Jupyter kernel runs.
- This affects the Python environment, not your project folder.
- No project files are created; only the environment changes (site‑packages).

After this, you typically restart the kernel once so everything is cleanly loaded.

---

### Cell 3 – Run Project A (Generator)

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
# OR
# !python D:\DigitalTwinsGeneratorGUI\generator\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

Let’s assume the version with `config.json` is used, since that’s the clean contract:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

**What happens conceptually inside `generator/app.py` (based on our design):**

1. **Config loading**
   - `app.py` reads `D:\DigitalTwinsGeneratorGUI\config.json`.
   - This config typically contains:
     - path to the telemetry file (e.g. `telemetry.parquet` or `telemetry.csv`)
     - generator parameters (rows per chunk, total rows, schema, etc.)
     - possibly socket settings for alerts.

2. **Telemetry generation**
   - The generator creates synthetic telemetry batches using `numpy`/`pandas`.
   - It writes them incrementally to the configured file:
     - e.g. `D:\DigitalTwinsGeneratorGUI\telemetry.parquet`
       or a subfolder like `D:\DigitalTwinsGeneratorGUI\data\telemetry.parquet`
       depending on how you set it in `config.json`.

3. **File behavior**
   - If the file doesn’t exist yet, it is created.
   - If it exists and your generator is configured to overwrite, it may truncate and start fresh.
   - If it appends, it will grow over time.

4. **Alerts (if implemented as we designed)**
   - After each chunk is written, the generator sends a small JSON alert over a socket:
     - e.g. `{ "event": "chunk_written", "timestamp": "...", "payload": {"rows": 10000} }`
   - When it finishes, it may send a `generation_complete` alert.

**Files created/modified:**

- **Telemetry file**  
  - e.g. `D:\DigitalTwinsGeneratorGUI\telemetry.parquet` (or `.csv`)  
  - This is the main artifact Project B will analyze.
- **Possibly `config.json`** if you created it manually beforehand; the generator usually just reads it, not writes it.

The notebook cell will show any console output from the generator (progress, logs, etc.) and then return when the generator finishes (or keep running if it’s a continuous stream).

---

### Cell 4 – Run Project B (Analyzer)

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py
# OR
# !python D:\DigitalTwinsGeneratorGUI\analyzer\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

Again, the clean version is:

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

**What happens inside `analyzer/app.py`:**

1. **Qt application startup**
   - A `QApplication` is created.
   - `MainWindow` is instantiated with `config_path="D:\DigitalTwinsGeneratorGUI\config.json"`.
   - The main GUI window appears on your desktop (separate from Jupyter).

2. **Config loading**
   - `MainWindow` reads the same `config.json`.
   - From this, it learns:
     - where the telemetry file is
     - which modules are enabled (statistics, clustering, forecasting, nlp, deep_learning, xai)
     - any socket/alert settings.

3. **Core components are created**
   Inside `MainWindow.__init__`, roughly:

   - **TelemetryReader**  
     - Knows how to read new chunks from the telemetry file (CSV/Parquet).
   - **FileMonitor**  
     - Tracks file size, row count, and detects resets.
   - **AlertListener**  
     - Listens on a socket for alerts from the generator (e.g. `chunk_written`).
   - **AnalyzerLoop**  
     - Periodically:
       - checks for new rows
       - reads them
       - runs selected analysis modules
       - sends results to the GUI.
   - **VisualizationTabs**  
     - Holds the plots for:
       - time series
       - clustering
       - forecasting
       - deep learning anomaly scores
       - NLP summary
       - XAI summary.
   - **HealthSummary / LogPanel**  
     - Shows health messages and logs (including alerts formatted by `AlertManager`).

4. **Event wiring**
   - `AlertListener` → when an alert arrives:
     - passes it to `AlertManager.add_alert()`
     - logs the formatted message
     - may trigger `HealthSummary` updates (e.g. “data updated”, “generation complete”).
   - `AnalyzerLoop` → on each cycle:
     - asks `TelemetryReader` for new rows
     - runs:
       - `statistics.run(df)`
       - `clustering.run(df)`
       - `forecasting.run(df)`
       - `nlp.run(df)`
       - `deep_learning.run(df)`
       - `xai.run(df)`
     - collects their outputs and health messages
     - passes them to `VisualizationTabs` and `HealthSummary`.

5. **GUI updates**
   - Plots are refreshed with:
     - time series curves
     - clustering scatter
     - forecast history + future
     - anomaly score curve
   - Text tabs show:
     - NLP keyword summary
     - XAI feature importance summary
   - Health panel shows warnings/errors (e.g. high anomaly ratio, unstable trend, error spikes).

**Files created/modified:**

- The Analyzer itself typically does **not** create new data files; it:
  - reads the telemetry file generated by Project A
  - may write logs if you configured logging to file (otherwise just console/GUI)
- It does not touch `config.json` except for reading.

The notebook cell that launched the Analyzer will stay “running” until you close the GUI window, because the Qt event loop is active in that process. But your Jupyter kernel is not blocked—it just spawned a separate process.

---

### Putting it all together: end‑to‑end flow

1. **You run Cell 1**  
   → Jupyter knows where your project lives.

2. **You run Cell 2**  
   → Environment is ready (all packages installed).

3. **You run Cell 3 (Generator)**  
   → Synthetic telemetry is generated and written to a file (e.g. `telemetry.parquet`) under `D:\DigitalTwinsGeneratorGUI\` (or a subfolder from `config.json`).  
   → Optionally, alerts are sent over a socket.

4. **You run Cell 4 (Analyzer)**  
   → A GUI window opens.  
   → It reads `config.json`, locates the telemetry file, and starts analyzing it.  
   → If the generator is still running and sending alerts, the Analyzer reacts in near real time.  
   → If the generator already finished, the Analyzer processes the existing file and shows static (but still interactive) analysis.



Let’s walk through the entire lifecycle in a clear, chronological way.

---

# 🧩 1. Our `config.json` (the contract between A and B)

Here is what our config means:

```json
{
  "file_path": "telemetry.parquet",
  "columns": ["Temperature", "Motor RPM", "Error Code"],
  "sampling_rate_hz": 10,
  "timestamp_format": "ISO8601",

  "output": {
    "file_path": "telemetry.parquet",
    "chunk_size": 10000
  },

  "alerts": {
    "socket_host": "127.0.0.1",
    "socket_port": 5050,
    "socket_enabled": true
  }
}
```

### 🔍 Key points:

- **Both `file_path` and `output.file_path` point to the same file**  
  → `telemetry.parquet`  
  → This file will be created in **D:\DigitalTwinsGeneratorGUI** (because paths are relative).

- **Generator writes telemetry in chunks of 10,000 rows**  
  → After each chunk, it sends a socket alert to the Analyzer.

- **Analyzer listens on 127.0.0.1:5050**  
  → When it receives `"chunk_written"`, it immediately refreshes.

- **Columns** define the schema of the generated data.

- **sampling_rate_hz = 10**  
  → The generator simulates 10 samples per second.

---

# 🧩 2. What happens when you run your Jupyter Notebook

Your notebook has four cells:

1. Add project root to `sys.path`
2. Install dependencies
3. Run Generator
4. Run Analyzer

Let’s walk through them.

---

# 🟦 **Cell 1 — sys.path setup**

```python
PROJECT_ROOT = r"D:\DigitalTwinsGeneratorGUI"
sys.path.append(PROJECT_ROOT)
```

### ✔ What happens

- Python can now import:
  - `generator.app`
  - `analyzer.app`
  - all submodules

### ❌ No files are created  
This is purely an in‑memory configuration.

---

# 🟦 **Cell 2 — pip install**

```python
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm
```

### ✔ What happens

- Our Jupyter environment installs all required packages.
- Nothing is written to your project folder.

### ❌ No project files created  
Only our Python environment changes.

---

# 🟦 **Cell 3 — Run Project A (Generator)**

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

or:

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

### ✔ What happens inside the Generator

1. **Reads config.json**  
   - Finds `"output.file_path": "telemetry.parquet"`

2. **Creates or overwrites `telemetry.parquet`**  
   Location:  
   ```
   D:\DigitalTwinsGeneratorGUI\telemetry.parquet
   ```

3. **Generates telemetry rows**  
   - Columns: Temperature, Motor RPM, Error Code  
   - Sampling rate: 10 Hz  
   - Writes in chunks of 10,000 rows

4. **After each chunk**  
   - Writes the chunk to the Parquet file  
   - Sends a socket alert:
     ```json
     {
       "event": "chunk_written",
       "timestamp": "...",
       "payload": { "rows": 10000 }
     }
     ```

5. **When finished**  
   - Sends:
     ```json
     { "event": "generation_complete" }
     ```

### 📁 Files created/modified

| File | Location | Description |
|------|----------|-------------|
| `telemetry.parquet` | `D:\DigitalTwinsGeneratorGUI\` | The generated telemetry file |
| (optional) logs | console only | No log files unless you added logging |

---

# 🟦 **Cell 4 — Run Project B (Analyzer)**

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py D:\DigitalTwinsGeneratorGUI\config.json
```

### ✔ What happens inside the Analyzer

1. **Qt GUI starts**  
   - A window opens on your desktop.

2. **Reads config.json**  
   - Learns that telemetry file is:
     ```
     telemetry.parquet
     ```
     → resolved to:
     ```
     D:\DigitalTwinsGeneratorGUI\telemetry.parquet
     ```

3. **Initializes core components**

   - **TelemetryReader**  
     Reads new rows from the Parquet file.

   - **FileMonitor**  
     Tracks file size and row count.

   - **AlertListener**  
     Opens a socket on 127.0.0.1:5050  
     Waits for alerts from the Generator.

   - **AnalyzerLoop**  
     Periodically:
     - checks for new rows
     - reads them
     - runs all analysis modules:
       - statistics
       - clustering
       - forecasting
       - NLP
       - deep_learning
       - xai
     - sends results to the GUI

   - **VisualizationTabs**  
     Updates plots and text summaries.

   - **HealthSummary**  
     Shows warnings and alerts.

4. **When a socket alert arrives**  
   Example: `"chunk_written"`

   - AlertListener receives it  
   - AlertManager formats it  
   - LogPanel displays it  
   - HealthSummary marks “data updated”  
   - AnalyzerLoop immediately reads the new chunk  
   - All analysis modules run on the new data  
   - GUI updates instantly

### 📁 Files created/modified

| File | Location | Description |
|------|----------|-------------|
| none | — | Analyzer does not create files unless you add logging |

Everything happens in memory + GUI.

---

# 🧩 3. End‑to‑end timeline (the whole system)

Here is the complete flow:

1. **You run the Generator**  
   → `telemetry.parquet` is created  
   → chunks are appended  
   → alerts are sent

2. **You run the Analyzer**  
   → GUI opens  
   → reads existing telemetry  
   → waits for alerts

3. **Generator writes a new chunk**  
   → file grows  
   → alert is sent  
   → Analyzer receives alert  
   → Analyzer reads new rows  
   → Analyzer runs all modules  
   → GUI updates plots and summaries  
   → HealthSummary shows status

4. **Generator finishes**  
   → sends `"generation_complete"`  
   → Analyzer marks generation as complete

---

# 🧩 4. What you will *see* when running the notebook

### In the Jupyter output:

- Generator logs (rows written, progress)
- Analyzer logs (alerts received, updates)

### On your desktop:

- A full GUI window with:
  - Time series plot
  - Clustering scatter
  - Forecast curve
  - Anomaly score curve
  - NLP summary
  - XAI summary
  - Health panel
  - Log panel

---


# Runner walkthrough

If **both the Generator and the Analyzer have their own GUIs**, then the execution flow from our Jupyter runner notebook becomes even clearer and more visual. Let me explain the *exact* chain of events, step by step, so one can know precisely what will appear, what files are created, and how the two GUIs interact.

I’ll describe the full lifecycle in a way that matches your folder structure, your `config.json`, and the architecture we built.

---

# ⭐ 1. Our Jupyter Notebook is just a launcher  
The notebook doesn’t *run* the GUIs inside itself — it simply **starts two separate desktop applications**:

- **Generator GUI**  
- **Analyzer GUI**

Each runs in its own process, with its own event loop, its own window, and its own workflow.

Our notebook remains free the whole time.

---

# ⭐ 2. What happens when you run the cells

Let’s go cell by cell.

---

# 🟦 **Cell 1 — Add project root to sys.path**

Nothing visible happens.  
No files are created.  
This just ensures Python can import your modules if needed.

---

# 🟦 **Cell 2 — Install dependencies**

Again, nothing visible in your project folder.  
Our Python environment gets the required packages.

---

# 🟦 **Cell 3 — Run the Generator GUI**

```python
!python D:\DigitalTwinsGeneratorGUI\generator\app.py
```

### ✔ What happens immediately  
A **desktop window opens** — the Generator GUI.

This GUI is responsible for:

- letting us configure generation parameters  
- starting/stopping the generation  
- showing progress  
- showing logs  
- sending alerts to the Analyzer  

### ✔ What files are created?

Our `config.json` says:

```json
"output": {
  "file_path": "telemetry.parquet",
  "chunk_size": 10000
}
```

Because the path is **relative**, the file is created in:

```
D:\DigitalTwinsGeneratorGUI\telemetry.parquet
```

This file grows as the Generator writes chunks of 10,000 rows.

### ✔ What else happens?

- After each chunk, the Generator GUI sends a socket alert to:
  ```
  127.0.0.1:5050
  ```
- The alert looks like:
  ```json
  { "event": "chunk_written", "payload": {"rows": 10000} }
  ```

The Generator GUI continues running until we stop it.

---

# 🟦 **Cell 4 — Run the Analyzer GUI**

```python
!python D:\DigitalTwinsGeneratorGUI\analyzer\app.py
```

### ✔ What happens immediately  
A **second desktop window opens** — the Analyzer GUI.

This GUI:

- loads `config.json`
- locates the telemetry file (`telemetry.parquet`)
- starts the AlertListener (listening on port 5050)
- starts the AnalyzerLoop
- initializes all visualization tabs
- initializes the health panel and log panel

### ✔ What files does the Analyzer create?

**None.**  
It only *reads* the telemetry file and *receives* alerts.

---

# ⭐ 3. Now the magic: how the two GUIs interact

Once both GUIs are running:

---

## 🔄 **Step A — Generator writes a chunk**

- It appends 10,000 new rows to `telemetry.parquet`
- It sends a socket alert `"chunk_written"`

---

## 🔄 **Step B — Analyzer receives the alert**

The Analyzer GUI:

1. Logs the alert in the Log Panel  
2. Marks “data updated” in the Health Summary  
3. Immediately reads the new rows from the file  
4. Runs all analysis modules:
   - statistics  
   - clustering  
   - forecasting  
   - NLP  
   - deep_learning  
   - XAI  
5. Updates all plots and summaries in the GUI

This happens **in real time**.

---

## 🔄 **Step C — Generator finishes**

It sends:

```json
{ "event": "generation_complete" }
```

The Analyzer GUI:

- logs it  
- marks generation as complete  
- stops expecting new chunks  

---

# ⭐ 4. What you will *see* on your screen

### 🖥️ Generator GUI window  
- Buttons: Start, Stop, Configure  
- Progress bar  
- Log output  
- Possibly a live preview of generated values  
- Status messages (“Chunk written”, “Generation complete”)

### 🖥️ Analyzer GUI window  
Tabs:

- **Time Series**  
  Live plot of Temperature, RPM, etc.

- **Clustering**  
  2D scatter of the last batch

- **Forecasting**  
  History + predicted future

- **Deep Learning**  
  Anomaly score curve

- **NLP**  
  Keyword summary from Error Code messages

- **XAI**  
  Feature importance for the target signal

Panels:

- **Health Summary**  
  Warnings like:
  - “High anomaly ratio”
  - “Unstable trend”
  - “Error keyword spike”

- **Log Panel**  
  Shows alerts from the Generator:
  ```
  [ALERT] chunk_written — rows=10000
  [ALERT] generation_complete
  ```

---

# ⭐ 5. Files created during the entire process

| File | Location | Created by | Description |
|------|----------|------------|-------------|
| `telemetry.parquet` | `D:\DigitalTwinsGeneratorGUI\` | Generator GUI | The actual telemetry data |
| `config.json` | `D:\DigitalTwinsGeneratorGUI\analyzer\core\...` | You | Read by both GUIs |
| (optional) logs | console only | Both | No log files unless you add logging |

No other files are created.

---

# ⭐ 6. Summary: What happens when you run the notebook

Here’s the whole story in one clean sequence:

1. Notebook prepares environment  
2. Notebook launches Generator GUI  
3. Notebook launches Analyzer GUI  
4. Generator GUI writes telemetry chunks  
5. Generator GUI sends alerts  
6. Analyzer GUI receives alerts  
7. Analyzer GUI reads new data  
8. Analyzer GUI updates all plots and health indicators  
9. When Generator finishes, Analyzer marks completion  

Our notebook is simply a **launcher**.  
The real action happens in the two GUIs.

---

Here is a **clear, visual, intuitive diagram** of how our entire system behaves when both GUIs are running.  
I will explain it in a way that mirrors exactly what happens on our machine, with your folder structure and our `config.json`.

---

# 🌐 **Full System Architecture Diagram (Generator GUI ↔ Analyzer GUI)**

Below is a clean, conceptual diagram of the data flow and event flow.

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                          JUPYTER NOTEBOOK (Runner)                           │
│                                                                              │
│  - Launches Generator GUI (separate process)                                 │
│  - Launches Analyzer GUI (separate process)                                  │
│  - Does NOT run the GUIs inside itself                                       │
└──────────────────────────────────────────────────────────────────────────────┘

                         │                                │
                         │                                │
                         ▼                                ▼

┌──────────────────────────────────────┐     ┌──────────────────────────────────┐
│          GENERATOR GUI               │     │          ANALYZER GUI            │
│  (D:\DigitalTwinsGeneratorGUI\...)   │     │  (D:\DigitalTwinsGeneratorGUI\...)│
│                                      │     │                                  │
│  • Reads config.json                 │     │  • Reads config.json             │
│  • Lets you start/stop generation    │     │  • Opens all visualization tabs  │
│  • Generates telemetry rows          │     │  • Starts AlertListener          │
│  • Writes chunks of 10,000 rows      │     │  • Starts AnalyzerLoop           │
│  • Sends socket alerts               │     │  • Monitors telemetry file       │
└──────────────────────────────────────┘     └──────────────────────────────────┘
                         │                                ▲
                         │ writes                         │ reads
                         ▼                                │
┌──────────────────────────────────────────────────────────────────────────────┐
│                         telemetry.parquet (shared file)                      │
│                         Location: D:\DigitalTwinsGeneratorGUI\               │
│                                                                              │
│  • Created by Generator GUI                                                  │
│  • Grows chunk-by-chunk (10,000 rows each)                                   │
│  • Analyzer GUI reads new rows as they appear                                │
└──────────────────────────────────────────────────────────────────────────────┘
                         │                                ▲
                         │ alerts                         │ reacts
                         ▼                                │
┌──────────────────────────────────────┐     ┌──────────────────────────────────┐
│        SOCKET ALERT CHANNEL          │     │         ALERT LISTENER           │
│      (127.0.0.1 : 5050 TCP)          │     │  (inside Analyzer GUI)           │
│                                      │     │                                  │
│  Generator sends:                    │     │  Analyzer receives:              │
│   • "chunk_written"                  │     │   • triggers immediate refresh   │
│   • "generation_complete"            │     │   • logs alert                   │
│                                      │     │   • updates HealthSummary        │
└──────────────────────────────────────┘     └──────────────────────────────────┘

                         │                                │
                         │ triggers                       │
                         ▼                                ▼

┌──────────────────────────────────────────────────────────────────────────────┐
│                           ANALYZER LOOP (inside GUI)                         │
│                                                                              │
│  On each alert OR timer tick:                                                │
│   • Reads new rows from telemetry.parquet                                    │
│   • Runs all analysis modules:                                               │
│        - statistics                                                          │
│        - clustering                                                          │
│        - forecasting                                                         │
│        - NLP                                                                 │
│        - deep_learning                                                       │
│        - xai                                                                 │
│   • Sends results to VisualizationTabs                                       │
│   • Sends warnings to HealthSummary                                          │
└──────────────────────────────────────────────────────────────────────────────┘

                         │
                         ▼

┌──────────────────────────────────────────────────────────────────────────────┐
│                           ANALYZER GUI VISUAL OUTPUT                         │
│                                                                              │
│  • Time Series Plot                                                          │
│  • Clustering Scatter                                                        │
│  • Forecast Curve                                                            │
│  • Anomaly Score Curve                                                       │
│  • NLP Keyword Summary                                                       │
│  • XAI Feature Importance Summary                                            │
│  • Health Summary Panel                                                      │
│  • Log Panel (shows alerts + internal logs)                                  │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

# 🧠 **What this diagram means in practice**

### ✔ Two GUIs run independently  
- Generator GUI produces data  
- Analyzer GUI consumes data  
- They communicate via:
  - a shared Parquet file  
  - a socket alert channel  

### ✔ The telemetry file is the “data backbone”  
- Always located at:  
  **D:\DigitalTwinsGeneratorGUI\telemetry.parquet**

### ✔ Alerts make the Analyzer reactive  
- Without alerts, it would still work (polling)  
- With alerts, it updates instantly

### ✔ The Analyzer GUI is a live dashboard  
It continuously:

- reads new data  
- analyzes it  
- visualizes it  
- warns you about anomalies  

### ✔ The Jupyter Notebook is only a launcher  
It does not run the GUIs inside itself.

---


# EXECUTE THE FOLLOWING THREE CELLS:

In [1]:
import sys
import os

# Path to your main folder
PROJECT_ROOT = r"D:\DigitalTwinsGeneratorGUI"

# Add to sys.path so Python can import generator and analyzer
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("Project root added to sys.path:", PROJECT_ROOT)

Project root added to sys.path: D:\DigitalTwinsGeneratorGUI


In [4]:
import os
os.chdir(r"D:\DigitalTwinsGeneratorGUI")

In [2]:
!pip install numpy pandas scikit-learn pyarrow PySide6 matplotlib scipy tqdm

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
print(os.path.abspath("config.json"))

D:\DigitalTwinsGeneratorGUI\config.json


In [44]:
import sys
print(sys.executable)

C:\miniforge3\envs\py311\python.exe


In [43]:
import subprocess

# Start Generator GUI in a true background process
subprocess.Popen(["python", "-m", "generator.app"])

# Start Analyzer GUI in a true background process
subprocess.Popen(["python", "-m", "analyzer.app", "config.json"])

<Popen: returncode: None args: ['python', '-m', 'analyzer.app', 'config.json']>